In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import os
import torch
import pickle
import config
import json

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from torch.utils.data import  DataLoader

import torch.nn.functional as F


from src.metric import *
from src.bi_encoder_training import cosent_loss,compute_batch_embeddings
from src.datasets import ResumeJDDataset

In [3]:
path=config.CLEANED_DATA_DIR

In [4]:
with open(os.path.join(path,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(path,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(path,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    
    

In [5]:
bi_encoder= SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2',device=config.device)
optimizer = torch.optim.AdamW(bi_encoder.parameters(), lr=2e-5)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
BATCH_SIZE=config.CHUNK_BATCH_SIZE

In [7]:
labels=[float(config.label_to_score[label]) for label in train_df['label']]
train_dataset=ResumeJDDataset(train_df['resume_text'].values,train_df['job_description_text'].values,labels)

train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)

In [8]:
min_delta=0.01
count=0
best_score=float('-inf')
epochs=3
patience=2

best_model_path=os.path.join(config.CHUNKED_MODEL_DIR,'bi_encoder_chunked')
os.makedirs(config.CHUNKED_MODEL_DIR,exist_ok=True)

In [12]:
for epoch in range(epochs):
    
    print(f"Epoch {epoch+1}/{epochs}")
    bi_encoder.train()
    total_loss=0
    
    progress_bar = tqdm(train_loader, desc="Training")
    
    for resumes,jds,labels in progress_bar:
        optimizer.zero_grad()
        
        resume_embs, jd_embs=compute_batch_embeddings(bi_encoder,resumes,jds)
        
        labels=labels.to(config.device)
        loss=cosent_loss(resume_embs,jd_embs,labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")
        
    average_loss=total_loss/len(progress_bar)
    print("Average Loss:",average_loss)
        
        
    #Validation    
    bi_encoder.eval()
    
    with torch.no_grad():
            
        val_resume_embs, val_jd_embs=compute_batch_embeddings(bi_encoder,val_df['resume_text'].values,val_df['job_description_text'].values)
        
        scores=F.cosine_similarity(val_resume_embs,val_jd_embs,dim=1)
        scores=scores.cpu().numpy()
        
        metrics=model_evaluation(scores,val_df,"job_description_text")
        print("NDCG:", metrics["ndcg_val"])
        print("MAP:", metrics["map_score"])
        
        final_score=0.6*metrics["ndcg_val"]+0.3*metrics["map_score"]+0.1*metrics["mrr_score"]
        
    if final_score>best_score+min_delta:
        best_score=final_score
        bi_encoder.save(best_model_path)
        count=0
    else:
        count+=1

    if count==patience:
        print("Early stopping triggered.")
        break
                
            
    

Epoch 1/3


Training:   0%|          | 0/780 [00:00<?, ?it/s]

Average Loss: 2.589313633243243
NDCG: 0.780187800239746
MAP: 0.878393627290483


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/3


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 3/3


Training:   0%|          | 0/780 [00:00<?, ?it/s]

Average Loss: 2.1236590508467112
NDCG: 0.8575381785435995
MAP: 0.9528506896259352


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
bi_encoder=SentenceTransformer(best_model_path,device=config.device)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
bi_encoder.eval()

with torch.no_grad():
        
    val_resume_embs, val_jd_embs=compute_batch_embeddings(bi_encoder,val_df['resume_text'].values,val_df['job_description_text'].values)
    
    scores=F.cosine_similarity(val_resume_embs,val_jd_embs,dim=1)
    scores=scores.cpu().numpy()
    
    metrics=model_evaluation(scores,val_df,"job_description_text")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (825 > 256). Running this sequence through the model will result in indexing errors


In [15]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.7129557367838034
Top-3 Accuracy: 1.0
NDCG: 0.8575381785435995
MAP: 0.9528506896259352
MRR: 0.9791666666666666


In [16]:
eval_df=val_df.copy()
eval_df['score']=scores
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('job_description_text'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads):.3f}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean():.3f}")


Score spread within groups:
Mean spread: 0.579
% groups with spread < 0.1:0.019


In [17]:
false_neg = eval_df[(eval_df['label'] == 2) & (eval_df['score'] < -0.3)]

print(f"Good Fit resumes scoring below -0.3: {len(false_neg)}")
print("\nSample false_neg resumes:")

i=0
for jd,group in false_neg.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Index:{row['index']}")
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

Good Fit resumes scoring below -0.3: 0

Sample false_neg resumes:


In [19]:
confused = eval_df[(eval_df['label'] == 1) & (eval_df['score'] < 1.2) & (eval_df['score'] > -0.1)]

print(f"confused predictions: {len(confused)}")
print("\nSample confused resumes:")

i=0
for jd,group in confused.groupby('job_description_text'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

confused predictions: 285

Sample confused resumes:
----------------------------------------------------------------------------------------------------
JD:  Experienced in Salesforce Industries Communications cloud.Certification a plus*10+ years in SFDC 5+ years of experience in Telecom domain solutioning for Quoteto Cash Architect software solutions usi
Score: 0.749
Resume: Professional SummaryBusiness Intelligence Consultant with a 10-year career in data warehousing, business intelligence reporting, and data management architecture. Progressive developer and technical team lead with a strength in design & development, as well as driving performance, reducing inefficie

Score: 0.717
Resume: SummaryExperienced Data Analyst who responds to shifting business needs and priorities in a systematic and effective way.  Excels at implementing operational assessments and conducting functional requirements analysis for businesses of all sized.  Committed to maintaining cutting edge technical sk

In [20]:

with torch.no_grad():
        
    test_resume_embs, test_jd_embs=compute_batch_embeddings(bi_encoder,test_df['resume_text'].values,test_df['job_description_text'].values)
    
    scores=F.cosine_similarity(test_resume_embs,test_jd_embs,dim=1)
    scores=scores.cpu().numpy()
    
    metrics=model_evaluation(scores,test_df,"job_description_text")

In [21]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.5078290407164832
Top-3 Accuracy: 0.9642857142857143
NDCG: 0.7042443083458614
MAP: 0.8357953771779312
MRR: 0.883879781420765


In [22]:
import json

In [23]:
results_dir=config.RESULTS_DIR
os.makedirs(results_dir,exist_ok=True)

with open(os.path.join(results_dir,'biencoder_chunks.json'),'w') as f:
    json.dump(metrics,f)